# 11 Market transfer: the required support in seven market conditions

**Purpose.** The paper computes its requirement objects in the reference market: the predicted
shadow price of the SMR mandate (Fig. 3), the required credit-rate fans and the statutory mask
(Fig. 4) and the common-window table (Table 2). This notebook carries those objects into seven
alternative *market conditions* and asks how far each one moves them. A market condition is one
set of ReEDS switches that changes the grid around the mandated fleet (gas price, demand,
renewable cost, transmission, a carbon cap) and leaves every nuclear input alone. Batch 3 re-solved
the same cost samples as the reference case under each condition, so the difference between a
condition's solved required rate and the reference prediction is the condition's effect alone.
The notebook estimates that difference as an additive shift per condition and online year, gives
it an error band, restates the requirement objects per condition, runs the registered gates and
the carbon-cap expectation test, and draws Fig. 5 (sources) and Supplementary Fig. 14.

**Inputs.**
- ReEDS output extracts `results/extracts/{smr100,step4,ratedesign,batch3}/{pvf_onm,cost_cap,
  cost_cap_fin_mult_noITC,cap_new_ann}.parquet` (notebook 05_extract_reeds_outputs).
- Shadow-price tables `results/exports/06a/{step3_checks,step4_checks}/duals_by_year.csv`
  (notebook 06a_run_tables_core) and `results/exports/06b/rate_design_checks/duals_by_year.csv`,
  `results/exports/06b/rate_design2_b3_checks/{duals_by_year,checks_summary}.csv`
  (notebook 06b_run_tables_rate_design).
- `results/exports/07/t09_required_itc.csv` (notebook 07_required_support): the reference-case
  exact required rate.
- `results/exports/01/mc_perdraw.npz` (notebook 01_monte_carlo_cost_engine): the 10,000 cost samples.
- The requirement map: `results/exports/10/m10b_population_rates.npz`, `m103_band_e7.csv`,
  `m104_requirement_fans{,_cut}.csv`, `m104c_shadow_price_fans.csv`, `m105_feasibility_mask{,_cut}.csv`,
  `m106_demonstration_gap{,_cut}.csv`, `m107_common_window_table.csv`,
  `m109_batch3_anchor_predictions.csv` (notebook 10_requirement_map).
- `data/registration/batch3/b3_run_spec.csv` (the frozen batch-3 design), the four casefiles
  `cases_nuclearlearning_{smr100,step4,ratedesign,batch3}.csv` and the fork's `inputs/financials`
  and `inputs/nuclear_learning` tables.

**Outputs** (all under `results/exports/11/`, internal names kept).
- Tables: `w00_status_flips.csv`, `w01_required_rate_perworld.csv`, `w02_world_year_shift.csv`,
  `w02b_shift_loo_cells.csv`, `w02c_band_e7w.csv`, `w03_requirement_fans_perworld{,_cut}.csv`,
  `w04_feasibility_mask_perworld{,_cut}.csv`, `w05_demonstration_gap_perworld{,_cut}.csv`,
  `w06_common_window_perworld.csv`, `w07_shadow_price_fans_perworld.csv`,
  `w08_co2cap_expectation.csv`, `w09_verdict.csv`, `market_transfer_metadata.json`.
- Panels: `y02_shadow_price_fans_perworld.png`, `y03_mask_perworld.png`,
  `y03_mask_perworld_si.png`, `y04_table2_perworld.png`, `y05_rate_fans_perworld_pct.png`,
  `y06_fig6_legend.png`.
- Composed figure: `results/figures/si/SF14_market_requirement_transfer.png`.

**Paper items.** Fig. 5 (panels y05, y03 and legend y06; notebook 16 composes it), Supplementary
Fig. 14, Supplementary Table 4 (sources w02, w04_cut, w07, w06, w08, w01, w09; notebook 16 writes
the table), Supplementary Note 8 (the registration and the verdict). Notebooks 12_flat_credit and
15_new_plant_value read `w01`.

**Runtime.** About one minute on the reference machine (4 threads).

In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import nlpaper
from nlpaper import paths, plotstyle as ps, h5io
nlpaper.setup(threads=4)

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


{'python': '3.12.3',
 'numpy': '1.26.4',
 'pandas': '2.2.2',
 'threads': 4,
 'threadpoolctl': True,
 'env_set': {'OMP_NUM_THREADS': '4',
  'MKL_NUM_THREADS': '4',
  'OPENBLAS_NUM_THREADS': '4',
  'NUMEXPR_NUM_THREADS': '4',
  'NUMBA_NUM_THREADS': '4'},
 'matplotlib': '3.8.4',
 'seed_note': None}

## 1. The runs, the market conditions and the reference inputs

**The seven market conditions.** Each condition is a switch set in the batch-3 casefile:

- **Low gas price** (`gaslo`) and **high gas price** (`gashi`): the AEO 2026 high and low
  oil-and-gas supply cases.
- **High electricity demand** (`demhi`): a high-electrification load profile.
- **Low-cost renewables** (`relo`) and **high-cost renewables** (`rehi`): the ATB advanced and
  conservative cost cases for wind, solar and storage.
- **Transmission-limited** (`translim`): no new interregional transmission.
- **National CO2 cap** (`co2cap`): a national carbon-dioxide cap that falls to 95% below the 2005
  level by 2050 (121 Mt in 2050).

The first six conditions are the ones of notebook 08 (Step 4 batch); the seventh is new in batch 3.

**The runs.** A *target trajectory* (`schedule` in the code) is a national path of mandated SMR
capacity from one published source. A *cost sample* (a `pct` or `draw_index`) is one joint Monte
Carlo draw of the nuclear cost world. Batch 3 block A re-solves, in each condition, the same cost
samples that the reference case solved: the p25 and p75 anchors of every target trajectory in the
six Step 4 conditions (12 runs each), and p05 to p95 anchors in the carbon-cap condition (30 runs).
That is 102 registered *paired samples*; 100 returned (the p75 anchors of `eo` and `iaea` in the
high-demand condition did not). "Paired" means the cost inputs of the condition run are the cost
inputs of its reference run, byte for byte (gate G0c re-proves it below), so only the market differs.

The notebook also re-derives the 126 earlier runs (18 reference SMR-program runs and their 108
Step 4 condition runs) and the 12 reference p25/p75 anchors (rate-design batch), which are the
reference sources of the batch-3 pairs.

**The requirement map.** Notebook 10 fits the *requirement map*, a regression that predicts the
required credit rate of any of the 10,000 cost samples in the reference market from its cost
inputs. `m10b` holds the map's population matrices (cost sample × online year per target
trajectory), `m109` its prediction at each batch-3 anchor, and `m103` its per-trajectory error band
E7 (the band that covers the map's leave-one-out error, in rate points).

The cell below reads every input, re-proves the registry (gate R0: the batch-3 run checks of
notebook 06b hold no FAIL), and rebuilds the financing constants the required-rate chain needs.
The discount-rate assert is a scientific invariant: the post-2026 ReEDS system real rate, 5.36%.

In [2]:
import json

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, PercentFormatter
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)

EXPORTS = paths.exports("11")          # every table and panel of this notebook
FIGURES = EXPORTS                      # the panels are sources for notebook 16 and for SF14 below
NB06A, NB06B = paths.exports("06a"), paths.exports("06b")
B3_SPEC = paths.require(paths.DATA / "registration" / "batch3" / "b3_run_spec.csv", "the frozen batch-3 registration (data/)")
PERDRAW = paths.require(paths.exports("01") / "mc_perdraw.npz", "notebook 01_monte_carlo_cost_engine")
CHECKS3 = NB06A / "step3_checks"
CHECKS4 = NB06A / "step4_checks"
RD_CHECKS = NB06B / "rate_design_checks"          # the 12 p25/p75 reference anchors' shadow prices
B3_CHECKS = NB06B / "rate_design2_b3_checks"
S3ANALYSIS = paths.exports("07")
RD2_EXPORTS = paths.exports("10")                 # the requirement map (notebook 10_requirement_map)
for _f in ("m10b_population_rates.npz", "m103_band_e7.csv", "m104_requirement_fans.csv", "m104_requirement_fans_cut.csv",
           "m104c_shadow_price_fans.csv", "m105_feasibility_mask.csv", "m105_feasibility_mask_cut.csv", "m106_demonstration_gap.csv",
           "m106_demonstration_gap_cut.csv", "m107_common_window_table.csv", "m109_batch3_anchor_predictions.csv"):
    paths.require(RD2_EXPORTS / _f, "notebook 10_requirement_map")
for _d, _nb in ((CHECKS3, "06a_run_tables_core"), (CHECKS4, "06a_run_tables_core"), (RD_CHECKS, "06b_run_tables_rate_design"),
                (B3_CHECKS, "06b_run_tables_rate_design")):
    paths.require(_d / "duals_by_year.csv", f"notebook {_nb}")
paths.require(S3ANALYSIS / "t09_required_itc.csv", "notebook 07_required_support")
FIN_DIR = paths.reeds_inputs() / "financials"
VERDICT = []              # (kind, name, result, detail) -> w09

# ---- registered constants (rate_design methods v2.4 / v2.8; market_transfer v2.0) --------------
TOL_SHORT, DISPLAY_CUT = 0.01, 0.03
CAPS = [0.50, 0.60]
W_PRIMARY, W_SENS, W_COMMON = 2035, 2033, 2038
QS = [5, 25, 50, 75, 95]
MODEL_YEARS = [2031, 2032, 2033, 2034, 2035, 2038, 2041, 2044, 2047, 2050]
GT_COV, GT_MAE = 0.85, 2.0            # GT1 / GT2 = the GM3 thresholds (v2.7.2)
WORLDS7 = ["gaslo", "gashi", "demhi", "relo", "rehi", "translim", "co2cap"]
WORLDS = ["base"] + WORLDS7
SENS6 = WORLDS7[:-1]
WLAB = {"base": "reference case", "gaslo": "low gas price", "gashi": "high gas price",
        "demhi": "high electricity demand", "relo": "low-cost renewables", "rehi": "high-cost renewables",
        "translim": "transmission-limited", "co2cap": "national CO2 cap (121 Mt in 2050)"}
SENS_C = dict(ps.SENS_C)
SENS_C["base"] = ps.BASE_C
assert "co2cap" in SENS_C
SCHEDULES = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
assert SCHEDULES == list(ps.SCHED_ORDER), ps.SCHED_ORDER
AB2TOK = {"eia": "eia_aeo_high", "aj": "abou_jaoude", "iaea": "iaea_high", "mck": "mckinsey", "cop28": "cop28", "eo": "eo2025"}
TOK2AB = {v: k for k, v in AB2TOK.items()}
GREY = "0.35"

# ---- R0: the batch-3 run checks (notebook 06b) hold no FAIL ---------------------------------
reg_checks = pd.read_csv(B3_CHECKS / "checks_summary.csv")
_fails = reg_checks[reg_checks["status"] == "FAIL"]
assert not len(_fails), _fails
VERDICT.append(("gate", "R0", "PASS", f"rate_design2_b3_checks registry: {reg_checks.status.value_counts().to_dict()}; no FAIL"))

# ---- the v2.9 map's population matrices (m10b) and its records --------------------------------
NPZ = np.load(RD2_EXPORTS / "m10b_population_rates.npz")
assert [str(s) for s in NPZ["SCHEDULES"]] == SCHEDULES
assert float(NPZ["TOL_SHORT"]) == TOL_SHORT and float(NPZ["DISPLAY_CUT"]) == DISPLAY_CUT
CHOSEN = str(NPZ["chosen"])
D2224 = float(NPZ["D2224"])
BUILD_TS = {ab: [int(t) for t in NPZ[f"BUILD_TS_{ab}"]] for ab in SCHEDULES}
IW0, IWRAW, SWRAW, CC, OCCW, OCC22, ZERO0, IREF, WPRICE, TOPK, BOTK, CSY = ({} for _ in range(12))
for ab in SCHEDULES:
    IW0[ab], IWRAW[ab], SWRAW[ab], CC[ab] = NPZ[f"IW_{ab}"], NPZ[f"IWRAW_{ab}"], NPZ[f"SWRAW_{ab}"], NPZ[f"CC_{ab}"]
    OCCW[ab], OCC22[ab], ZERO0[ab] = NPZ[f"OCCW_{ab}"], NPZ[f"OCC22_{ab}"], NPZ[f"ZERO_{ab}"]
    IREF[ab], WPRICE[ab], TOPK[ab], BOTK[ab], CSY[ab] = NPZ[f"IREF_{ab}"], NPZ[f"WPRICE_{ab}"], NPZ[f"TOPKNOT_{ab}"], NPZ[f"BOTKNOT_{ab}"], NPZ[f"CSY_{ab}"]
    assert np.array_equal(IW0[ab], np.where(ZERO0[ab], 0.0, IWRAW[ab]))
    assert all(t in MODEL_YEARS for t in BUILD_TS[ab])
N_DRAWS = IW0["eia"].shape[0]
E7_PTS = {ab: int(v) for ab, v in zip(SCHEDULES, NPZ["E7_PTS"])}
E7 = {ab: E7_PTS[ab] / 100.0 for ab in SCHEDULES}
P50_DRAW = {ab: int(v) for ab, v in zip(SCHEDULES, NPZ["P50_DRAW"])}
CENSORED = {(s.split(":")[0], int(s.split(":")[1])) for s in NPZ["CENSORED"].astype(str)}
m103 = pd.read_csv(RD2_EXPORTS / "m103_band_e7.csv")
assert {ab: int(v) for ab, v in m103[m103["ab"] != "pooled"].set_index("ab")["E7_pts"].items()} == E7_PTS
# the v2.7 map's band, a record column of w01 (ST4 f) and of the w09 gate text (ST4 g); m103 carries it
E7_V27 = {ab: int(v) for ab, v in m103[m103["ab"] != "pooled"].set_index("ab")["E7_pts_v27"].items()}
m109 = pd.read_csv(RD2_EXPORTS / "m109_batch3_anchor_predictions.csv")
assert m109["run"].nunique() == 102 and len(m109) == 935
M109 = m109.set_index(["run", "t"])

# ---- the MC ensemble (for the world slack rule: each anchor's cost in each year) ----------------
Z = np.load(PERDRAW)
YI = {int(y): i for i, y in enumerate(Z["years"])}

# ---- the cases: the 126 v1.0 cases (GT0) + the 100 returned batch-3 anchors ----------------------
spec = pd.read_csv(B3_SPEC).set_index("run")
specA = spec[spec["block"].isin(["A1", "A2"])]
duals_b3 = pd.read_csv(B3_CHECKS / "duals_by_year.csv")
NEW_CASES = sorted(duals_b3["case"].unique())
assert len(NEW_CASES) == 100 and set(NEW_CASES) <= set(specA.index)
NOT_RETURNED = sorted(set(specA.index) - set(NEW_CASES))
assert NOT_RETURNED == ["smr100_eo_p75_demhi", "smr100_iaea_p75_demhi"], NOT_RETURNED
PCTS3 = ["p05", "p50", "p95"]
BASE_CASES = [f"smr100_{ab}_{p}" for ab in SCHEDULES for p in PCTS3]
SENS_CASES = [f"{c}_{s}" for s in SENS6 for c in BASE_CASES]
BASE2_CASES = [f"smr100_{ab}_{p}" for ab in SCHEDULES for p in ["p25", "p75"]]     # the batch-3 A1 sources (rate_design batch)
OLD_CASES = BASE_CASES + SENS_CASES
CASES = OLD_CASES + BASE2_CASES + NEW_CASES
cases_csv = {b: pd.read_csv(paths.casefile(b), index_col=0) for b in ("smr100", "step4", "batch3", "ratedesign")}
CF_OF = {c: ("smr100" if c in BASE_CASES else "step4" if c in SENS_CASES else "ratedesign" if c in BASE2_CASES else "batch3") for c in CASES}

def parse_case(c):
    parts = c.split("_")
    assert parts[0] == "smr100", c
    return parts[1], parts[2], (parts[3] if len(parts) > 3 else "base")

def base_of(c):
    return "_".join(c.split("_")[:3])

def sw(case, name):
    csv_ = cases_csv[CF_OF[case]]
    v = csv_.loc[name, case]
    if pd.isna(v) or str(v).strip() == "":
        v = csv_.loc[name, "Default Value"]
    return str(v).strip()

for c in CASES:
    assert sw(c, "GSw_NuclearCapMandateTechScen") == "smr", c
TECH = "nuclear-smr"
SCEN = {c: sw(c, "GSw_NuclearCapMandateScen") for c in CASES}
for c in SENS_CASES + NEW_CASES:
    assert SCEN[c] == SCEN[base_of(c)], c
# ReEDS outputs come from the parquet extracts of notebook 05 (one file per batch and key)
_EXTRACTS = {}
def load(case, key):
    """The rows of one run in one ReEDS output table (columns as in the h5 file)."""
    b = CF_OF[case]
    if (b, key) not in _EXTRACTS:
        _EXTRACTS[(b, key)] = {c: g.drop(columns="case").reset_index(drop=True) for c, g in h5io.load(b, key).groupby("case", sort=False)}
    return _EXTRACTS[(b, key)][case]
_present = {b: set(h5io.load(b, "pvf_onm")["case"]) for b in sorted(set(CF_OF.values()))}
missing = [c for c in CASES if c not in _present[CF_OF[c]]]
assert not missing, f"runs missing from the extracts: {missing[:5]}"

# ---- price level, discount rate, financing (v1.0 setup, verbatim) ------------------------------
DEFL = pd.read_csv(FIN_DIR / "deflator.csv")
DEFL.columns = ["t", "Deflator"]
DEFL = DEFL.set_index("t")["Deflator"]
TO2024 = 1.0 / float(DEFL.loc[2024])
YEARS = np.arange(2010, 2051)
sys_fin = pd.read_csv(FIN_DIR / "financials_sys_ATB2024.csv")
infl = pd.read_csv(FIN_DIR / "inflation_default.csv")
sys_fin = sys_fin.merge(infl, on="t", how="left")
sys_fin["d_nom"] = ((1 - sys_fin["debt_fraction"]) * (sys_fin["rroe_nom"] - 1)
                    + sys_fin["debt_fraction"] * (sys_fin["interest_rate_nom"] - 1) * (1 - sys_fin["tax_rate"]) + 1)
sys_fin["d_real"] = sys_fin["d_nom"] / sys_fin["inflation_rate"]
def on_years(col):
    s = sys_fin.set_index("t")[col].reindex(range(1990, YEARS[-1] + 1)).ffill()
    return s.loc[YEARS].to_numpy(float)
D_REAL, D_NOM, TAX, IB = on_years("d_real"), on_years("d_nom"), on_years("tax_rate"), on_years("interest_rate_nom")
DR = float(np.mean(D_REAL[YEARS >= 2026]))   # 2026-09-22: post-2026 ReEDS system real rate (5.36%); was the 2010-2050 mean (4.55%)
assert round(DR, 4) == 1.0536, DR
_infl = infl.set_index("t")["inflation_rate"]
assert abs(float(_infl.loc[2023] * _infl.loc[2024]) - D2224) < 1e-12
def yi(t):
    return int(t - YEARS[0])

# ---- the duals: base (step3_checks), the six worlds (step4_checks), batch 3 (b3 checks) ----------
duals3 = pd.read_csv(CHECKS3 / "duals_by_year.csv")
duals4 = pd.read_csv(CHECKS4 / "duals_by_year.csv")
DCOLS = ["case", "t", "mandate_MW", "cap_MW", "slack_MW", "dual_2004_MWyr", "dual_raw", "dual_2024_MWyr"]
duals_rd = pd.read_csv(RD_CHECKS / "duals_by_year.csv")
DUALS = pd.concat([duals3[duals3["case"].isin(BASE_CASES)][DCOLS], duals4[duals4["case"].isin(SENS_CASES)][DCOLS],
                   duals_rd[duals_rd["case"].isin(BASE2_CASES)][DCOLS], duals_b3[DCOLS]], ignore_index=True)
assert set(DUALS["case"]) == set(CASES)
DUALS["mandated"] = DUALS["mandate_MW"] > 0
DUALS["binding"] = DUALS["dual_2004_MWyr"] > 1.0
YEARS_RUN = sorted(int(t) for t in DUALS["t"].unique())
assert len(YEARS_RUN) == 16, YEARS_RUN
TPREV = {t: (YEARS_RUN[i - 1] if i > 0 else None) for i, t in enumerate(YEARS_RUN)}
DCASE = {c: g.set_index("t").sort_index() for c, g in DUALS.groupby("case")}
for c in NEW_CASES:
    assert np.array_equal(DCASE[c]["mandate_MW"].to_numpy(), DCASE[base_of(c)]["mandate_MW"].to_numpy()), c

# ---- the base-world exact rates and statuses of every source anchor (t09, frozen) ----------------
t09 = pd.read_csv(S3ANALYSIS / "t09_required_itc.csv")
t09i = t09.set_index(["case", "t"])
for c in NEW_CASES:
    assert base_of(c) in set(t09["case"]), c
print(f"cases: {len(CASES)} = 18 + 12 base + 108 step-4 + {len(NEW_CASES)} batch-3 anchors (not returned: {NOT_RETURNED}); "
      f"map {CHOSEN}; E7 {E7_PTS}; solve years {YEARS_RUN[0]}-{YEARS_RUN[-1]}")

cases: 238 = 18 + 12 base + 108 step-4 + 100 batch-3 anchors (not returned: ['smr100_eo_p75_demhi', 'smr100_iaea_p75_demhi']); map M5X(lam=0.1); E7 {'eia': 6, 'aj': 4, 'iaea': 2, 'mck': 2, 'cop28': 2, 'eo': 1}; solve years 2010-2050


## 2. w01: the exact required rate on every run

**What it computes.** For each run and solve year in which the mandate binds (its *shadow price*,
the marginal system cost of one more MW of required SMR capacity, is positive) and new SMR plants
are built, the chain converts the shadow price into the *Equivalent ITC*: the uniform investment
tax-credit rate that gives the same support to the building region that needs the most (the
*headline rate*), grossed up for the tax-equity haircut. This is the t09 procedure of notebook 07,
ported unchanged. Rows where the mandate is slack, where the mandate does not increase, or where no
new plant is built carry a status instead of a rate.

**Why re-derive it.** The condition runs have their own shadow prices and building regions, so the
rate must be computed from each run's own outputs. The financing inputs (capital cost and
financing multiplier) are identical to the reference run's (gate G0c).

**Gate GT0.** The ported chain must reproduce the reference-case t09 table of notebook 07 on every
source run it covers: the 18 reference SMR-program runs and the 12 p25/p75 anchors (identical
status strings, headline rate within the 3-decimal rounding of t09).

**Reading the output.** `w01` has one row per gated run-year: status, the solved headline rate
(`i_model_headline`, a fraction), and for batch-3 rows the reference-case exact rate
(`rate_base_exact`), the map's reference prediction (`rate_pred_base`, from m109, with the earlier
map versions' predictions beside it as records) and `paired_rate` (both runs carry a rate in a model
year). `w00` lists the batch-3 anchor-years whose status differs from the reference run.

In [3]:
DEP = pd.read_csv(FIN_DIR / "depreciation_schedules_default.csv")
CS = pd.read_csv(FIN_DIR / "construction_schedules_mc.csv")
CC_EXPS = np.append([0.0], np.arange(0.5, 10.5, 1.0))
assert len(CS) == len(CC_EXPS), (len(CS), len(CC_EXPS))

def ccmult_reeds(sched_name, ib):
    x = pd.to_numeric(CS[str(sched_name)], errors="coerce").fillna(0.0).to_numpy()
    return 1.0 + float(np.sum(x * (ib ** CC_EXPS - 1.0)))

def fin_tech(case):
    fin = sw(case, "financials_tech_suffix")
    ft = pd.read_csv(FIN_DIR / f"financials_tech_{fin}.csv")
    ft.columns = [c.lstrip("*") for c in ft.columns]
    return ft[ft["i"] == "Nuclear-SMR"]

_NF_CACHE = {}
def natfin(case):
    key = sw(case, "financials_tech_suffix")
    if key in _NF_CACHE:
        return _NF_CACHE[key]
    grp = fin_tech(case).set_index("t").reindex(YEARS).ffill().bfill()
    dep_col = str(int(grp["depreciation_sch"].iloc[0]))
    dep_frac = pd.to_numeric(DEP[dep_col], errors="coerce").fillna(0.0).to_numpy(float)
    pv_dep = np.array([np.sum(dep_frac / dn ** np.arange(1, len(dep_frac) + 1)) for dn in D_NOM])
    eval_p = float(grp["eval_period"].iloc[0])
    risk = 1.0 + grp["finance_diff_real"].to_numpy(float) * ((1 - (1 / D_REAL) ** eval_p) / (D_REAL - 1.0))
    sys_pvf = (1 - (1 / D_REAL) ** (30 - 1)) / (D_REAL - 1.0) + 1
    tech_pvf = (1 - (1 / D_REAL) ** (eval_p - 1)) / (D_REAL - 1.0) + 1
    eval_adj = sys_pvf / tech_pvf
    ccm = np.array([ccmult_reeds(grp.loc[t, "construction_sch"], IB[yi(t)]) for t in YEARS])
    natbase = pd.Series(ccm / (1 - TAX) * (1 - TAX * pv_dep) * risk * eval_adj, index=YEARS)
    k = pd.Series((1 - TAX * pv_dep) / (1 - TAX * pv_dep / 2), index=YEARS)
    ccm = pd.Series(ccm, index=YEARS)
    _NF_CACHE[key] = (natbase, k, ccm)
    return natbase, k, ccm

inc_ob = pd.read_csv(FIN_DIR / "incentives_obbba.csv")
nuc_ob = inc_ob[inc_ob["i"].str.upper().str.startswith("NUCLEAR")]
PEN = float(nuc_ob["itc_tax_equity_penalty"].iloc[0])
assert (nuc_ob["itc_tax_equity_penalty"] == PEN).all()
NL_DIR = paths.reeds_inputs() / "nuclear_learning"
def trajectory(scen):
    tr = pd.read_csv(NL_DIR / f"nuclear_cap_trajectory_{scen}.csv")
    tr.columns = [c.lstrip("*") for c in tr.columns]
    return tr.set_index(tr.columns[0])["MW"]
TRAJ = {c: trajectory(SCEN[c]) for c in CASES}

COSTCAP, FINMULT, CAPNEW, PVF = {}, {}, {}, {}
for c in CASES:
    PVF[c] = load(c, "pvf_onm").set_index("t")["Value"]
    cc = load(c, "cost_cap")
    COSTCAP[c] = cc[cc["i"] == TECH].groupby("t")["Value"].mean()
    fmn = load(c, "cost_cap_fin_mult_noITC")
    FINMULT[c] = fmn[fmn["i"] == TECH].set_index(["r", "t"])["Value"]
    cn = load(c, "cap_new_ann")
    CAPNEW[c] = cn[cn["i"] == TECH].set_index(["r", "t"])["Value"]
g0c_fail = [c for c in SENS_CASES + NEW_CASES
            if not (COSTCAP[c].equals(COSTCAP[base_of(c)]) and FINMULT[c].sort_index().equals(FINMULT[base_of(c)].sort_index()))]
assert not g0c_fail, g0c_fail
VERDICT.append(("gate", "G0c", "PASS", "cost_cap and cost_cap_fin_mult_noITC identical to the base-world source for all 208 world cases (108 step-4 + 100 batch-3)"))

rows = []
for c in CASES:
    ab, pct, world = parse_case(c)
    nbse, K, CCM = natfin(c)
    d = DCASE[c]
    cc, fm, bl, tr = COSTCAP[c], FINMULT[c], CAPNEW[c], TRAJ[c]
    for t in YEARS_RUN:
        if not bool(d.loc[t, "mandated"]):
            continue
        dual_pos = bool(d.loc[t, "binding"])
        m_inc = float(tr.get(t, 0.0)) - float(tr.get(TPREV[t], 0.0) if TPREV[t] else 0.0)
        blt = bl.xs(t, level="t") if t in bl.index.get_level_values("t") else pd.Series(dtype=float)
        blt = blt[blt > 0]
        if not dual_pos:
            status = "slack: required ITC is zero"
        elif m_inc <= 0:
            status = "no mandate increase: no build receives a credit"
        elif not len(blt):
            status = "no new builds: dual prices retention, not a build"
        else:
            status = "rate"
        row = dict(case=c, base_case=base_of(c), batch=("batch3" if c in NEW_CASES else "base_p25p75" if c in BASE2_CASES else "v1"), schedule=ab, pct=pct, world=world, t=t,
                   status=status, S_t_2004_per_MW=None, n_build_regions=0, fm_min=None, fm_min_region=None, ccmult=None,
                   i_model_min=None, i_model_max=None, i_model_headline=None, i_model_headline_raw=None, itc_insufficient=False)
        if status == "rate":
            S = float(d.loc[t, "dual_raw"])
            S_alt = float(d.loc[t, "dual_2004_MWyr"]) * float(PVF[c].loc[t])
            assert abs(S - S_alt) <= 1e-4 * max(1.0, abs(S)), (c, t, S, S_alt)
            fmb = fm.xs(t, level="t").reindex(blt.index).dropna()
            m_r = S / (float(cc.loc[t]) * fmb) * float(K.loc[t])
            i_r = m_r / (1.0 - PEN)
            row.update(S_t_2004_per_MW=round(S), n_build_regions=len(i_r), fm_min=float(fmb.min()), fm_min_region=str(fmb.idxmin()),
                       ccmult=round(float(CCM.loc[t]), 4), i_model_min=round(float(i_r.min()), 3), i_model_max=round(float(i_r.max()), 3),
                       i_model_headline=round(float(i_r.max()), 3), i_model_headline_raw=float(i_r.max()), itc_insufficient=bool(i_r.max() > 1.0))
        rows.append(row)
w01 = pd.DataFrame(rows)

# GT0: the ported chain reproduces the reference-case t09 (notebook 07) on every source run t09 covers:
# the 18 reference SMR-program runs and the 12 p25/p75 anchors (statuses identical, headline within the 3-dp rounding)
key = ["case", "t"]
GT0_CASES = BASE_CASES + BASE2_CASES
m2 = w01[w01["case"].isin(GT0_CASES)].merge(t09[t09["case"].isin(GT0_CASES)][key + ["status", "i_model_headline"]], on=key, suffixes=("", "_t09"), how="outer", indicator=True)
assert (m2["_merge"] == "both").all(), m2[m2["_merge"] != "both"][key]
assert (m2["status"] == m2["status_t09"]).all(), "t09 status mismatch"
rr2 = m2[m2["status"] == "rate"]
hd2 = (rr2["i_model_headline"] - rr2["i_model_headline_t09"]).abs()
assert hd2.max() <= 5e-4, hd2.max()
VERDICT.append(("gate", "GT0", "PASS", f"the ported chain reproduces t09 (notebook 07) on the {len(GT0_CASES)} reference source runs "
                f"(18 SMR-program runs + 12 p25/p75 anchors): {len(m2)} gated rows, statuses identical, headline max |diff| {hd2.max():.1e} "
                f"({int((hd2 == 0).sum())}/{len(rr2)} exact at 3 dp); the 108 Step 4 condition runs go through the same code"))

# the base-world exact rate and status (t09), and the map's predictions (m109; v2.8 m99 and v2.7 m89 beside) on the batch-3 rows
def base_status(c, t):
    try:
        return str(t09i.loc[(base_of(c), t), "status"])
    except KeyError:
        return "not gated"
w01["base_status"] = [base_status(c, t) for c, t in zip(w01["case"], w01["t"])]
w01["rate_base_exact"] = [float(t09i.loc[(base_of(c), t), "i_model_headline"]) if bs == "rate" else np.nan
                          for c, t, bs in zip(w01["case"], w01["t"], w01["base_status"])]
for col_out, col_in in [("rate_pred_base", "rate_pred_base"), ("S_kw_pred_base", "S_kw_pred_base"), ("zeroed_base", "zeroed"), ("E7_pts", "E7_pts"),
                        ("rate_pred_base_v28", "rate_pred_base_v28"), ("E7_pts_v28", "E7_pts_v28"), ("rate_pred_base_v27", "rate_pred_base_v27")]:
    w01[col_out] = [M109.loc[(c, t), col_in] if (c in NEW_CASES and (c, t) in M109.index) else np.nan for c, t in zip(w01["case"], w01["t"])]
w01["E7_pts_v27"] = w01["schedule"].map(E7_V27).where(w01["batch"] == "batch3")
_chk = w01[(w01["batch"] == "batch3") & (w01["base_status"] == "rate")]
assert float((_chk["rate_base_exact"] - [float(M109.loc[(c, t), "solved_base_rate"]) for c, t in zip(_chk["case"], _chk["t"])]).abs().max()) < 1e-9
w01["dual_2004_MWyr"] = [float(DCASE[c].loc[t, "dual_2004_MWyr"]) for c, t in zip(w01["case"], w01["t"])]
w01["paired_rate"] = (w01["status"] == "rate") & (w01["base_status"] == "rate") & w01["t"].isin(MODEL_YEARS)
w01["i_model_headline_raw"] = w01["i_model_headline_raw"].astype(float)
w01.to_csv(EXPORTS / "w01_required_rate_perworld.csv", index=False)

# w00: the anchor inventory + status flips (batch 3)
b3rows = w01[w01["batch"] == "batch3"]
flips = b3rows[(b3rows["status"] != b3rows["base_status"]) & (b3rows["base_status"] != "not gated")]
w00 = flips[["case", "base_case", "world", "schedule", "pct", "t", "base_status", "status", "dual_2004_MWyr"]].copy()
w00.to_csv(EXPORTS / "w00_status_flips.csv", index=False)
n_rate_w = b3rows[b3rows["status"] == "rate"].groupby("world").size().reindex(WORLDS7).fillna(0).astype(int)
all_slack = sorted(c for c, g in b3rows[b3rows["t"].isin(MODEL_YEARS)].groupby("case") if (g["status"] != "rate").all())
_ga_missing = sorted(set(NEW_CASES) - set(b3rows[b3rows["t"].isin(MODEL_YEARS)]["case"]))
VERDICT.append(("gate", "GA-w", "PASS" if not _ga_missing else "FAIL", (f"anchors without a gated model-year row: {_ga_missing}; " if _ga_missing else "")
                + f"every returned anchor yields a required-rate row set; rate rows per world {n_rate_w.to_dict()}; "
                f"anchors slack (or otherwise non-rate) in every model year: {all_slack or 'none'} (reported, not dropped)"))
VERDICT.append(("quoting", "status-flips", "DISCLOSE",
                f"{len(w00)} of {len(b3rows)} batch-3 anchor-years change t09 status vs the base world: "
                f"{int((w00['base_status'] == 'rate').sum())} base rate-years are not rate in the world, {int((w00['status'] == 'rate').sum())} rate rows exist only in the world; per world: "
                + "; ".join(f"{w} {int((w00['world'] == w).sum())}" for w in WORLDS7)))
print(f"w01: {len(w01)} gated rows; batch-3 rate rows per world: {n_rate_w.to_dict()}; paired rate cells: {int(b3rows['paired_rate'].sum())}")
print(f"w00: {len(w00)} status flips")

w01: 2180 gated rows; batch-3 rate rows per world: {'gaslo': 110, 'gashi': 107, 'demhi': 90, 'relo': 110, 'rehi': 110, 'translim': 110, 'co2cap': 257}; paired rate cells: 894
w00: 20 status flips


## 3. w02: the additive condition-by-online-year shift and its band

**What it computes.** For condition w and model year t, the shift δ is the mean, over the
condition's *paired rate cells* (anchor-years with a rate status in both the reference run and the
condition run), of the condition's solved rate minus the map's reference prediction (m109). The
shift is *additive*: it moves every cost sample's predicted rate by the same number of rate points
in that condition and year.

**Why paired samples.** Each condition run shares its cost sample with a reference run, so the
difference isolates the market effect; the map's own cost dependence cancels. An unpaired
comparison would mix cost and market effects.

**The band.** Leaving one anchor run out at a time gives the error of the mean (the
*leave-one-out* error, LOO). Per condition and trajectory the band is
E7w = ceil(√(E7² + q90²)), where q90 is the 90th percentile of the absolute LOO residual pooled over
the condition's cells.

**The gates.** GT1 asks that at least 85% of a condition's cells lie within ±E7w of their LOO
prediction (coverage); GT2 asks that the pooled LOO mean absolute error be at most 2 rate points.
PASS means both hold; **PARTIAL** means exactly one holds, and here it is always GT1: the band covers
the cells, but a single shift per condition and year leaves more than 2 points of error, so each
condition carries its own wider band E7w rather than the map's E7. FAIL would mean neither holds.

**The kills.** KTW1 fires for a condition whose q90 exceeds twice the widest E7 (12 points); such a
condition is quoted from its exact solved anchors only and its restated objects carry
`restated = False`. It fires for the high gas price. KTW2 asks whether a finer
condition × trajectory × year shift would halve the error; it fires nowhere. The multiplicative
(ratio) shift and the residuals against the earlier map versions (v2.8, v2.7, carried in m109) are
recorded, not used.

In [4]:
P = w01[(w01["batch"] == "batch3") & w01["paired_rate"]].copy()
P["d"] = 100.0 * (P["i_model_headline_raw"] - P["rate_pred_base"])              # rate points
P["d_v28"] = 100.0 * (P["i_model_headline_raw"] - P["rate_pred_base_v28"])
P["d_v27"] = 100.0 * (P["i_model_headline_raw"] - P["rate_pred_base_v27"])
grp = P.groupby(["world", "t"])
for col in ("d", "d_v28", "d_v27"):
    S_, N_ = grp[col].transform("sum"), grp[col].transform("size")
    P[f"delta_{col}"] = S_ / N_
    P[f"delta_loo_{col}"] = np.where(N_ > 1, (S_ - P[col]) / (N_ - 1).clip(lower=1), np.nan)
    P[f"e_{col}"] = P[col] - P[f"delta_loo_{col}"]
P["n_cells"] = grp["d"].transform("size")
# KTW2 companion: a world x schedule x year shift (leave-one-out within the schedule's cells)
g2 = P.groupby(["world", "schedule", "t"])["d"]
S2, N2 = g2.transform("sum"), g2.transform("size")
P["delta_loo_sched"] = np.where(N2 > 1, (S2 - P["d"]) / (N2 - 1).clip(lower=1), np.nan)
P["e_sched"] = P["d"] - P["delta_loo_sched"]
# the ratio companion (record only): geometric-mean factor over the paired cells with a positive prediction
P["lr"] = np.log(P["i_model_headline_raw"] / P["rate_pred_base"].clip(lower=1e-6))
g3 = P.groupby(["world", "t"])["lr"]
S3, N3 = g3.transform("sum"), g3.transform("size")
P["m_loo"] = np.exp(np.where(N3 > 1, (S3 - P["lr"]) / (N3 - 1).clip(lower=1), np.nan))
P["e_ratio"] = 100.0 * (P["i_model_headline_raw"] - P["rate_pred_base"] * P["m_loo"])

# bands per world x schedule, coverage, gates
rows2c, Q90, E7W = [], {}, {}
for w in WORLDS7:
    pw = P[P["world"] == w]
    e = pw["e_d"].dropna()
    Q90[w] = float(np.quantile(e.abs(), 0.9)) if len(e) else np.nan
    E7W[w] = {ab: int(np.ceil(np.sqrt(E7_PTS[ab] ** 2 + Q90[w] ** 2))) for ab in SCHEDULES}
    for ab in SCHEDULES:
        es = pw[pw["schedule"] == ab]["e_d"].dropna()
        rows2c.append(dict(world=w, schedule=ab, E7_pts=E7_PTS[ab], q90_world=Q90[w], E7w_pts=E7W[w][ab], n_cells=int(len(es)),
                           mae_sched=float(es.abs().mean()) if len(es) else np.nan,
                           cov_E7w_sched=float((es.abs() <= E7W[w][ab]).mean()) if len(es) else np.nan,
                           cov_E7_sched=float((es.abs() <= E7_PTS[ab]).mean()) if len(es) else np.nan))
w02c = pd.DataFrame(rows2c)
w02c.to_csv(EXPORTS / "w02c_band_e7w.csv", index=False)
P["E7w_pts"] = [E7W[w][ab] for w, ab in zip(P["world"], P["schedule"])]
P["within_E7w"] = P["e_d"].abs() <= P["E7w_pts"]
P["within_E7_v28"] = P["e_d_v28"].abs() <= P["E7_pts_v28"]
P["within_E7_v27"] = P["e_d_v27"].abs() <= P["E7_pts_v27"]
w02b = P[["case", "world", "schedule", "pct", "t", "i_model_headline_raw", "rate_base_exact", "rate_pred_base", "rate_pred_base_v28", "rate_pred_base_v27",
          "d", "delta_d", "delta_loo_d", "e_d", "E7_pts", "E7w_pts", "within_E7w", "d_v28", "e_d_v28", "within_E7_v28", "d_v27", "e_d_v27", "within_E7_v27",
          "delta_loo_sched", "e_sched", "m_loo", "e_ratio", "n_cells"]].copy()
w02b.to_csv(EXPORTS / "w02b_shift_loo_cells.csv", index=False)

rows2, GT = [], {}
for w in WORLDS7:
    pw = P[P["world"] == w]
    e, e28, e27 = pw["e_d"].dropna(), pw["e_d_v28"].dropna(), pw["e_d_v27"].dropna()
    es = pw["e_sched"].dropna()
    er = pw["e_ratio"].dropna()
    mae, cov = float(e.abs().mean()), float(pw["within_E7w"].mean())
    gt1, gt2 = cov >= GT_COV, mae <= GT_MAE
    GT[w] = dict(mae=mae, cov=cov, p90=Q90[w], gt1=gt1, gt2=gt2, n=int(len(e)), n_runs=int(pw["case"].nunique()),
                 verdict=("PASS" if gt1 and gt2 else "PARTIAL" if (gt1 or gt2) else "FAIL"),
                 ktw1=bool(Q90[w] > 2 * max(E7_PTS.values())),
                 mae_sched=float(es.abs().mean()) if len(es) else np.nan, n_sched=int(len(es)),
                 mae_same_cells=float(pw.loc[es.index, "e_d"].abs().mean()) if len(es) else np.nan,
                 mae_v28=float(e28.abs().mean()), cov_v28=float(pw["within_E7_v28"].mean()),
                 mae_v27=float(e27.abs().mean()), cov_v27=float(pw["within_E7_v27"].mean()),
                 mae_ratio=float(er.abs().mean()) if len(er) else np.nan)
    GT[w]["ktw2"] = bool(np.isfinite(GT[w]["mae_sched"]) and GT[w]["mae_sched"] < 0.5 * GT[w]["mae_same_cells"])
    for t in MODEL_YEARS:
        pt = pw[pw["t"] == t]
        rows2.append(dict(world=w, t=t, n_cells=int(len(pt)), n_runs=int(pt["case"].nunique()),
                          delta_pts=float(pt["d"].mean()) if len(pt) else np.nan,
                          delta_v28_pts=float(pt["d_v28"].mean()) if len(pt) else np.nan,
                          delta_v27_pts=float(pt["d_v27"].mean()) if len(pt) else np.nan,
                          loo_mae_pts=float(pt["e_d"].abs().mean()) if len(pt) and pt["e_d"].notna().any() else np.nan,
                          loo_p90_pts=float(np.quantile(pt["e_d"].dropna().abs(), 0.9)) if pt["e_d"].notna().any() else np.nan,
                          sd_pts=float(pt["d"].std(ddof=1)) if len(pt) > 1 else np.nan,
                          ratio_factor=float(np.exp(pt["lr"].mean())) if len(pt) else np.nan,
                          no_shift=bool(len(pt) == 0)))
w02 = pd.DataFrame(rows2)
w02["q90_world"] = w02["world"].map(Q90)
w02["E7w_pts_by_schedule"] = w02["world"].map(lambda w: ";".join(f"{ab}={E7W[w][ab]}" for ab in SCHEDULES))
w02["GT1"] = w02["world"].map(lambda w: "PASS" if GT[w]["gt1"] else "FAIL")
w02["GT2"] = w02["world"].map(lambda w: "PASS" if GT[w]["gt2"] else "FAIL")
w02.to_csv(EXPORTS / "w02_world_year_shift.csv", index=False)
DELTA = {(r.world, int(r.t)): (0.0 if r.no_shift else float(r.delta_pts) / 100.0) for r in w02.itertuples()}
NO_SHIFT = sorted((r.world, int(r.t)) for r in w02.itertuples() if r.no_shift)
NO_SHIFT_USED = sorted((w, t) for (w, t) in NO_SHIFT if any(t in BUILD_TS[ab] for ab in SCHEDULES))

for w in WORLDS7:
    g = GT[w]
    VERDICT.append(("gate", f"GT1/GT2 {w}", g["verdict"],
                    f"{WLAB[w]}: {g['n']} paired rate cells from {g['n_runs']} anchors; leave-one-out MAE {g['mae']:.2f} pts (GT2 <= {GT_MAE}), "
                    f"coverage within E7w {g['cov']:.3f} (GT1 >= {GT_COV}); q90 {g['p90']:.2f}; E7w " + ", ".join(f"{ab} {E7W[w][ab]}" for ab in SCHEDULES)
                    + f"; the same residuals vs v2.8 m99: MAE {g['mae_v28']:.2f}, coverage at the bare v2.8 E7 (no shift band) {g['cov_v28']:.3f}; vs v2.7 m89: MAE {g['mae_v27']:.2f}, coverage at the bare v2.7 E7 {g['cov_v27']:.3f}"))
    VERDICT.append(("kill", f"KTW1 {w}", "EXACT-ANCHORS-ONLY" if g["ktw1"] else "PASS",
                    f"q90 of the leave-one-out residual {g['p90']:.2f} pts vs 2 x max E7 = {2 * max(E7_PTS.values())}"))
    VERDICT.append(("kill", f"KTW2 {w}", "DISCLOSE-COARSE" if g["ktw2"] else "PASS",
                    f"world x schedule x year companion: leave-one-out MAE {g['mae_sched']:.2f} on {g['n_sched']} cells with a within-schedule fold vs "
                    f"{g['mae_same_cells']:.2f} for the world x year shift on the same cells (fires below half)"))
VERDICT.append(("record", "ratio-companion", "record only",
                "the multiplicative (ratio) shift on the base prediction, leave-one-out MAE per world: " + ", ".join(f"{w} {GT[w]['mae_ratio']:.2f}" for w in WORLDS7)
                + "; the additive shift: " + ", ".join(f"{w} {GT[w]['mae']:.2f}" for w in WORLDS7)))
VERDICT.append(("quoting", "no-shift", "DISCLOSE" if NO_SHIFT_USED else "none",
                f"(world, year) pairs with no paired rate cell (the world's objects keep the base map's value there, flagged): {NO_SHIFT or 'none'}; "
                f"of which build years: {NO_SHIFT_USED or 'none'}"))
KTW1_WORLDS = [w for w in WORLDS7 if GT[w]["ktw1"]]
print(w02.pivot(index="t", columns="world", values="delta_pts").round(2).reindex(columns=WORLDS7).to_string())
print()
print(pd.DataFrame(GT).T[["n", "n_runs", "mae", "p90", "cov", "verdict", "ktw1", "ktw2", "mae_sched", "mae_v28", "cov_v28", "mae_v27", "cov_v27", "mae_ratio"]].to_string())
print("E7w by world x schedule:"); print(pd.DataFrame(E7W).T.reindex(WORLDS7).to_string())

world  gaslo  gashi  demhi   relo   rehi  translim  co2cap
t                                                         
2031    9.42 -11.19   0.68   4.10  -1.71      0.36    0.17
2032   17.11 -15.13  -2.42   5.60  -1.71      0.44    1.08
2033   10.52  -9.52  -5.67   6.09  -8.58     -6.01    0.47
2034    9.63 -13.15  -4.80   3.80  -8.04     -4.05    0.16
2035   11.96 -16.37  -4.13   7.47  -5.44     -0.40    0.27
2038   12.12 -23.42  -6.23   6.71  -7.02     -2.45    0.13
2041   13.76 -22.56 -12.44  10.26  -9.76    -11.19   -5.00
2044   20.06 -27.75 -12.17   9.15  -9.43     -4.11  -10.85
2047   16.13 -24.03 -12.68  10.35 -11.33     -8.72  -20.10
2050   16.86 -26.60  -9.47   9.63  -9.99     -4.27  -24.73

            n n_runs       mae        p90       cov  verdict   ktw1   ktw2 mae_sched   mae_v28   cov_v28   mae_v27   cov_v27  mae_ratio
gaslo     110     12  5.271578  11.153825  0.909091  PARTIAL  False  False  5.408554  5.304272  0.354545  5.364005  0.463636   6.958584
gashi     107     1

## 4. w03 to w07: the requirement objects per condition

**What it computes.** For each condition, the rate matrix is the map's population matrix (10,000
cost samples × online years per target trajectory, before the slack rule) plus δ, clipped to
[0, 1.2]. The condition's own *slack rule* then zeroes every sample-year whose cost is at or below
the cost of the condition's most expensive slack anchor in that year (the mandate would not bind).
The shadow price moves with the rate through the same cell-wise conversion factor. The fans (w03),
the statutory mask (w04), the demonstration gap (w05), the common-window table (w06) and the
shadow-price fans (w07) are then the notebook-10 cells with the condition's band E7w in place of E7.

**Terms.** The *statutory ceiling* (`cap`) is the credit rate the law allows (50%; 60% as a
sensitivity). The *allowance* (`cut`) is the tolerance before a sample counts as above the ceiling:
the registered one-point allowance (`reg`, 0.01) or the three-point display allowance (`disp`,
0.03; the `_cut` files). The *common window* ends in 2038. A sample is *credit-feasible from year t
on* when its required rate is at or below the ceiling plus allowance in t and every later online
year.

**Gate GT0b.** With δ = 0, the reference slack rule and E7, the code must reproduce notebook 10's
m104, m104_cut, m104c, m105 (+cut), m106 (+cut) and m107 exactly.

**Reading the output.** Every share is a share of the drawn cost ensemble under the stated ceiling
and allowance. It is not a probability that a program succeeds. Rows of a KTW1 condition carry
`restated = False`.

In [5]:
ANCH_W = {}          # (world, ab) -> list of (case, draw_index) among the returned anchors
for c in NEW_CASES:
    ab, pct, w = parse_case(c)
    ANCH_W.setdefault((w, ab), []).append((c, int(specA.loc[c, "draw_index"])))
STATUS_W = w01.set_index(["case", "t"])["status"]

def zero_mask_world(w, ab):
    T = len(BUILD_TS[ab])
    zmask = np.zeros((N_DRAWS, T), bool)
    if w == "base":
        return ZERO0[ab].copy()
    occ = Z[f"occ_{AB2TOK[ab]}_smr"]
    for j, t in enumerate(BUILD_TS[ab]):
        slack = [d for c, d in ANCH_W[(w, ab)] if (c, t) in STATUS_W.index and str(STATUS_W.loc[(c, t)]).startswith("slack")]
        if slack:
            cut = max(occ[d, YI[t]] for d in slack)
            zmask[:, j] = OCC22[ab][:, j] <= cut
    return zmask

def world_matrices(w):
    IWw, SWw, ZW = {}, {}, {}
    for ab in SCHEDULES:
        d = np.array([DELTA.get((w, t), 0.0) if w != "base" else 0.0 for t in BUILD_TS[ab]])
        r = np.clip(IWRAW[ab] + d[None, :], 0.0, 1.2)
        s = np.clip(SWRAW[ab] + d[None, :] * CC[ab] * OCCW[ab] / CSY[ab][None, :], 0.0, None)
        zmask = zero_mask_world(w, ab)
        r[zmask] = 0.0
        s[zmask] = 0.0
        IWw[ab], SWw[ab], ZW[ab] = r, s, zmask
    return IWw, SWw, ZW

BAND = {"base": 0, "lo": -1, "hi": +1}

def requirement_fans(IWw, ZW, E7w, w, cut=None):
    fan_rows = []
    for ab in SCHEDULES:
        occ_ref = OCCW[ab][P50_DRAW[ab]]
        dref = np.array([DELTA.get((w, t), 0.0) if w != "base" else 0.0 for t in BUILD_TS[ab]])
        for band, sign in BAND.items():
            iwm = np.clip(IWw[ab] + sign * E7w[ab], 0.0, None); iwm[ZW[ab]] = 0.0
            refm = np.clip(IREF[ab] + dref + sign * E7w[ab], 0.0, None)
            if cut is None:
                needm = np.maximum(iwm - TOL_SHORT, 0.0) * OCCW[ab]
                pairs = [("rate", iwm, refm), ("need_kw", needm, np.maximum(refm - TOL_SHORT, 0.0) * occ_ref)]
            else:
                ratec = np.maximum(iwm - cut, 0.0); refc = np.maximum(refm - cut, 0.0)
                pairs = [("rate", ratec, refc), ("need_kw", ratec * OCCW[ab], refc * occ_ref)]
            for qty, arr, ref in pairs:
                Pq = np.percentile(arr, QS, axis=0)
                for j, t in enumerate(BUILD_TS[ab]):
                    fan_rows.append(dict(schedule=ab, band=band, quantity=qty, year=t,
                                         **{f"p{q:02d}": round(float(Pq[i, j]), 4) for i, q in enumerate(QS)},
                                         ref=round(float(ref[j]), 4), censored=(ab, t) in CENSORED,
                                         above_top_pct=round(float(1 - TOPK[ab][j]) * 100, 1), below_bot_pct=round(float(BOTK[ab][j]) * 100, 1),
                                         p95_clamp_pinned=False, band_pts=int(round(E7w[ab] * 100))))
    return pd.DataFrame(fan_rows)

def feasibility_tables(IWw, ZW, E7w, cut):
    mask_rows, world_rows, gap_rows = [], [], []
    for ab in SCHEDULES:
        for band, sign in BAND.items():
            iwm = np.clip(IWw[ab] + sign * E7w[ab], 0.0, None); iwm[ZW[ab]] = 0.0
            for cap in CAPS:
                above = iwm > cap + cut
                for j, t in enumerate(BUILD_TS[ab]):
                    mask_rows.append(dict(schedule=ab, band=band, cap=cap, year=t, share_above=round(float(above[:, j].mean()), 4),
                                          share_above_from=round(float(above[:, j:].any(axis=1).mean()), 4), censored=(ab, t) in CENSORED))
                for wname, wend in [("w2035", W_PRIMARY), ("w2033", W_SENS), ("w2038", W_COMMON)]:
                    post = [j for j, t in enumerate(BUILD_TS[ab]) if t > wend]
                    infeas = np.any(above[:, post], axis=1) if post else np.zeros(N_DRAWS, bool)
                    world_rows.append(dict(schedule=ab, band=band, cap=cap, window=wname, share_not_credit_feasible=round(float(infeas.mean()), 4)))
                    win = [j for j, t in enumerate(BUILD_TS[ab]) if t <= wend]
                    gap_kw = np.maximum(iwm[:, win] - cut - cap, 0.0) * OCCW[ab][:, win]
                    gap_B = gap_kw @ WPRICE[ab][win] if win else np.zeros(N_DRAWS)
                    gq = np.percentile(gap_B, QS)
                    gap_rows.append(dict(schedule=ab, band=band, cap=cap, window=wname, window_applicable=bool(win),
                                         share_needing_demo=round(float((gap_B > 0).mean()), 4),
                                         **{f"gap_p{q:02d}_B": round(float(gq[i]), 3) for i, q in enumerate(QS)}, gap_mean_B=round(float(gap_B.mean()), 3)))
    u51 = pd.DataFrame(mask_rows); u51w = pd.DataFrame(world_rows)
    u51["kind"] = "per_year"; u51w["kind"] = "per_world"
    return pd.concat([u51, u51w], ignore_index=True), pd.DataFrame(gap_rows)

def common_window_table(IWw, ZW, E7w):
    rows36 = []
    for ab in SCHEDULES:
        ts = np.array(BUILD_TS[ab]); T = len(ts)
        for cutname, cut in {"reg": TOL_SHORT, "disp": DISPLAY_CUT}.items():
            for band, sign in BAND.items():
                iwm = np.clip(IWw[ab] + sign * E7w[ab], 0.0, None); iwm[ZW[ab]] = 0.0
                for cap in CAPS:
                    above = iwm > cap + cut
                    anyab = above.any(axis=1)
                    last_j = np.where(anyab, T - 1 - np.argmax(above[:, ::-1], axis=1), -1)
                    ly = np.where(anyab, ts[np.clip(last_j, 0, T - 1)], 0)
                    never = ly == ts[-1]
                    ever = (~never) & (ly > 0)
                    rows36.append(dict(schedule=ab, cap=cap, band=band, cut=cutname, common_year=W_COMMON, first_build_year=int(ts[0]),
                                       share_under_from_start=round(float((ly == 0).mean()), 4), share_under_by_common=round(float((ly <= W_COMMON).mean()), 4),
                                       share_graduating_after_common=round(float((ever & (ly > W_COMMON)).mean()), 4), share_never=round(float(never.mean()), 4),
                                       share_ever_after_grants=round(float(ever.mean()), 4), complete_year=(int(ly[ever].max()) if ever.any() else -1),
                                       complete_year_q90=(int(np.quantile(ly[ever], 0.9, method="inverted_cdf")) if ever.any() else -1),
                                       complete_year_q95=(int(np.quantile(ly[ever], 0.95, method="inverted_cdf")) if ever.any() else -1),
                                       complete_year_q99=(int(np.quantile(ly[ever], 0.99, method="inverted_cdf")) if ever.any() else -1)))
    return pd.DataFrame(rows36)

def shadow_price_fans(SWw, ZW, w):
    rows_s = []
    for ab in SCHEDULES:
        S = SWw[ab]
        Pq = np.percentile(S, QS, axis=0)
        for j, t in enumerate(BUILD_TS[ab]):
            ref = np.nan
            if w == "base":
                ref = float(M109x.get((f"smr100_{ab}_p50", t), np.nan))
            elif (w, ab) in ANCH_W:
                c50 = f"smr100_{ab}_p50_{w}"
                if (c50, t) in STATUS_W.index and STATUS_W.loc[(c50, t)] == "rate":
                    ref = float(w01.set_index(["case", "t"]).loc[(c50, t), "S_t_2004_per_MW"]) * TO2024 / 1000.0
            rows_s.append(dict(schedule=ab, band="base", quantity="S_kw", year=t, **{f"p{q:02d}": round(float(Pq[i, j]), 2) for i, q in enumerate(QS)},
                               mean=round(float(S[:, j].mean()), 2), ref=round(ref, 2) if np.isfinite(ref) else np.nan, censored=(ab, t) in CENSORED,
                               above_top_pct=round(float(1 - TOPK[ab][j]) * 100, 1), below_bot_pct=round(float(BOTK[ab][j]) * 100, 1), n_zeroed=int(ZW[ab][:, j].sum())))
    return pd.DataFrame(rows_s)

# the base-world S reference of m104c (the solved p50 world's exact S) read back from the m104c file
_m104c = pd.read_csv(RD2_EXPORTS / "m104c_shadow_price_fans.csv")
M109x = {(f"smr100_{r.schedule}_p50", int(r.year)): float(r.ref) for r in _m104c.itertuples()}

# GT0b: delta = 0, the base slack rule, E7 -> m104 / m104_cut / m104c / m105 / m106 / m107 exactly
IWb, SWb, ZWb = world_matrices("base")
for ab in SCHEDULES:
    assert np.array_equal(IWb[ab], IW0[ab]) and np.array_equal(ZWb[ab], ZERO0[ab])
    assert np.allclose(SWb[ab], np.where(ZERO0[ab], 0.0, SWRAW[ab]), atol=1e-12)
E7b = {ab: E7[ab] for ab in SCHEDULES}
def same(a, b, name):
    b = b[a.columns]
    for col in a.columns:
        if a[col].dtype.kind in "fc":
            ok = np.allclose(a[col].astype(float).to_numpy(), b[col].astype(float).to_numpy(), atol=1e-9, equal_nan=True)
        else:
            ok = (a[col].astype(str).to_numpy() == b[col].astype(str).to_numpy()).all()
        assert ok, (name, col)
same(requirement_fans(IWb, ZWb, E7b, "base"), pd.read_csv(RD2_EXPORTS / "m104_requirement_fans.csv"), "m104")
same(requirement_fans(IWb, ZWb, E7b, "base", DISPLAY_CUT), pd.read_csv(RD2_EXPORTS / "m104_requirement_fans_cut.csv"), "m104_cut")
_f5, _g6 = feasibility_tables(IWb, ZWb, E7b, TOL_SHORT)
same(_f5, pd.read_csv(RD2_EXPORTS / "m105_feasibility_mask.csv"), "m105"); same(_g6, pd.read_csv(RD2_EXPORTS / "m106_demonstration_gap.csv"), "m106")
_f5c, _g6c = feasibility_tables(IWb, ZWb, E7b, DISPLAY_CUT)
same(_f5c, pd.read_csv(RD2_EXPORTS / "m105_feasibility_mask_cut.csv"), "m105_cut"); same(_g6c, pd.read_csv(RD2_EXPORTS / "m106_demonstration_gap_cut.csv"), "m106_cut")
same(common_window_table(IWb, ZWb, E7b), pd.read_csv(RD2_EXPORTS / "m107_common_window_table.csv"), "m107")
same(shadow_price_fans(SWb, ZWb, "base"), _m104c, "m104c")
VERDICT.append(("gate", "GT0b", "PASS", "with delta = 0, the base slack rule and E7 the copied cells reproduce m104 (+cut), m104c, m105 (+cut), m106 (+cut) and m107 exactly"))

# the seven worlds (+ base) -> w03..w07
F3, F3c, M4, M4c, G5, G5c, T6, S7 = [], [], [], [], [], [], [], []
ZCOUNT = {}
for w in WORLDS:
    IWw, SWw, ZW = world_matrices(w)
    E7w = E7b if w == "base" else {ab: E7W[w][ab] / 100.0 for ab in SCHEDULES}
    ZCOUNT[w] = {ab: int(ZW[ab].sum()) for ab in SCHEDULES}
    restated = w == "base" or w not in KTW1_WORLDS
    for lst, df in ((F3, requirement_fans(IWw, ZW, E7w, w)), (F3c, requirement_fans(IWw, ZW, E7w, w, DISPLAY_CUT)), (S7, shadow_price_fans(SWw, ZW, w))):
        df.insert(0, "world", w); df["restated"] = restated; lst.append(df)
    for cut, (lm, lg) in (((TOL_SHORT), (M4, G5)), ((DISPLAY_CUT), (M4c, G5c))):
        f_, g_ = feasibility_tables(IWw, ZW, E7w, cut)
        f_.insert(0, "world", w); g_.insert(0, "world", w); f_["restated"] = restated; g_["restated"] = restated
        lm.append(f_); lg.append(g_)
    t_ = common_window_table(IWw, ZW, E7w); t_.insert(0, "world", w); t_["restated"] = restated; T6.append(t_)
w03, w03c = pd.concat(F3, ignore_index=True), pd.concat(F3c, ignore_index=True)
w04, w04c = pd.concat(M4, ignore_index=True), pd.concat(M4c, ignore_index=True)
w05, w05c = pd.concat(G5, ignore_index=True), pd.concat(G5c, ignore_index=True)
w06, w07 = pd.concat(T6, ignore_index=True), pd.concat(S7, ignore_index=True)
for df in (w03, w03c, w04, w04c, w05, w05c, w06, w07):
    df["no_shift_flag"] = ([((w, int(y)) in NO_SHIFT) if pd.notna(y) else False for w, y in zip(df["world"], df["year"])] if "year" in df.columns else False)
w03.to_csv(EXPORTS / "w03_requirement_fans_perworld.csv", index=False); w03c.to_csv(EXPORTS / "w03_requirement_fans_perworld_cut.csv", index=False)
w04.to_csv(EXPORTS / "w04_feasibility_mask_perworld.csv", index=False); w04c.to_csv(EXPORTS / "w04_feasibility_mask_perworld_cut.csv", index=False)
w05.to_csv(EXPORTS / "w05_demonstration_gap_perworld.csv", index=False); w05c.to_csv(EXPORTS / "w05_demonstration_gap_perworld_cut.csv", index=False)
w06.to_csv(EXPORTS / "w06_common_window_perworld.csv", index=False)
w07.to_csv(EXPORTS / "w07_shadow_price_fans_perworld.csv", index=False)
VERDICT.append(("record", "world-slack-rule", "applied", "world-years zeroed by each world's own slack anchors: " + "; ".join(f"{w} {sum(ZCOUNT[w].values())}" for w in WORLDS)
                + f" (base: the v2.9 rule, {sum(ZCOUNT['base'].values())})"))

# the readings: Table 2 per world (50% cap, display cut, base band), the Fig 4b end labels, the 2050 median S
t2 = w06[(w06["cap"] == 0.5) & (w06["cut"] == "disp") & (w06["band"] == "base")].pivot(index="schedule", columns="world", values="share_never").reindex(SCHEDULES)[WORLDS]
t2u = w06[(w06["cap"] == 0.5) & (w06["cut"] == "disp") & (w06["band"] == "base")].pivot(index="schedule", columns="world", values="share_under_by_common").reindex(SCHEDULES)[WORLDS]
print("Table 2 per world -- never credit-feasible share, 50% cap, display cut, base band:"); print(t2.round(3).to_string())
print("\nTable 2 per world -- share under the cap by the common window (2038):"); print(t2u.round(3).to_string())
s50 = w07[w07["year"] == 2050].pivot(index="schedule", columns="world", values="p50").reindex(SCHEDULES)[WORLDS]
print("\nmedian predicted shadow price in 2050 (2024 $/kW) per world:"); print(s50.round(0).to_string())
env = w04[(w04["kind"] == "per_world") & (w04["window"] == "w2038") & (w04["cap"] == 0.5)].groupby("world")["share_not_credit_feasible"].agg(["min", "max"]).reindex(WORLDS)
_rw = ["base"] + [w for w in WORLDS7 if w not in KTW1_WORLDS]
VERDICT.append(("quoting", "envelope-2038-50pct", f"{len(_rw) - 1} restated worlds",
                "post-2038 not-credit-feasible share at the 50% cap, min-max over schedules and band ends (the map restatement; KTW1 worlds omitted): "
                + "; ".join(f"{w} {env.loc[w, 'min']:.2f}-{env.loc[w, 'max']:.2f}" for w in _rw)
                + f"; over the restated worlds {env.loc[_rw[1:], 'min'].min():.2f}-{env.loc[_rw[1:], 'max'].max():.2f}"
                + (f"; exact-anchors-only (KTW1), not restated: {KTW1_WORLDS}" if KTW1_WORLDS else "")))

Table 2 per world -- never credit-feasible share, 50% cap, display cut, base band:
world      base  gaslo  gashi  demhi   relo   rehi  translim  co2cap
schedule                                                            
eia       0.530  0.775  0.006  0.338  0.682  0.328     0.449   0.028
aj        0.738  0.901  0.255  0.601  0.844  0.592     0.678   0.295
iaea      0.684  0.849  0.230  0.552  0.787  0.545     0.629   0.268
mck       0.722  0.860  0.275  0.596  0.810  0.588     0.667   0.314
cop28     0.821  0.917  0.495  0.734  0.884  0.730     0.786   0.524
eo        0.886  0.953  0.625  0.819  0.931  0.815     0.858   0.652

Table 2 per world -- share under the cap by the common window (2038):
world      base  gaslo  gashi  demhi   relo   rehi  translim  co2cap
schedule                                                            
eia       0.350  0.170  0.802  0.576  0.209  0.520     0.544   0.433
aj        0.259  0.084  0.745  0.399  0.145  0.408     0.322   0.417
iaea      0.316  0

## 5. w08: the carbon-cap expectation test (EC)

The registered expectation (2026-09-15) is that the carbon cap is slack in the demonstration window
and binds after 2038. The window column of Table 2 (the share of samples at or below the ceiling by
2038) should therefore move less than the never-reached column (decided by the late-horizon rate).
Test EC compares the two moves against the reference case for every trajectory and ceiling at the
display allowance; it passes when the window moves less in every cell. The registered-allowance
count is recorded beside it.

In [6]:
rows8 = []
for cut in ("disp", "reg"):
    b = w06[(w06["world"] == "base") & (w06["cut"] == cut) & (w06["band"] == "base")].set_index(["schedule", "cap"])
    c = w06[(w06["world"] == "co2cap") & (w06["cut"] == cut) & (w06["band"] == "base")].set_index(["schedule", "cap"])
    for (ab, cap) in b.index:
        d_never = float(c.loc[(ab, cap), "share_never"] - b.loc[(ab, cap), "share_never"])
        d_under = float(c.loc[(ab, cap), "share_under_by_common"] - b.loc[(ab, cap), "share_under_by_common"])
        rows8.append(dict(cut=cut, schedule=ab, cap=cap, never_base=float(b.loc[(ab, cap), "share_never"]), never_co2cap=float(c.loc[(ab, cap), "share_never"]),
                          under_by_2038_base=float(b.loc[(ab, cap), "share_under_by_common"]), under_by_2038_co2cap=float(c.loc[(ab, cap), "share_under_by_common"]),
                          d_never=d_never, d_under_by_2038=d_under, window_moves_less=abs(d_under) < abs(d_never), never_falls=d_never < 0))
w08 = pd.DataFrame(rows8)
w08.to_csv(EXPORTS / "w08_co2cap_expectation.csv", index=False)
ec_disp = w08[w08["cut"] == "disp"]
EC = bool(ec_disp["window_moves_less"].all())
VERDICT.append(("test", "EC", "PASS" if EC else "NOT-MET",
                f"CO2-cap world vs base (display cut, base band): |d share_under_by_2038| < |d share_never| in {int(ec_disp['window_moves_less'].sum())} of {len(ec_disp)} schedule x cap cells; "
                f"never-under share falls in {int(ec_disp['never_falls'].sum())} of {len(ec_disp)}; d never (50% cap) " + ", ".join(f"{r.schedule} {r.d_never:+.3f}" for r in ec_disp[ec_disp['cap'] == 0.5].itertuples())
                + "; d under-by-2038 (50% cap) " + ", ".join(f"{r.schedule} {r.d_under_by_2038:+.3f}" for r in ec_disp[ec_disp['cap'] == 0.5].itertuples())
                + f"; registered cut: {int(w08[w08['cut'] == 'reg']['window_moves_less'].sum())} of {len(ec_disp)}"))
print(w08[w08["cut"] == "disp"].round(3).to_string(index=False))

 cut schedule  cap  never_base  never_co2cap  under_by_2038_base  under_by_2038_co2cap  d_never  d_under_by_2038  window_moves_less  never_falls
disp      eia  0.5       0.530         0.028               0.350                 0.433   -0.502            0.083               True         True
disp      eia  0.6       0.327         0.000               0.525                 0.633   -0.327            0.108               True         True
disp       aj  0.5       0.738         0.295               0.259                 0.417   -0.443            0.157               True         True
disp       aj  0.6       0.592         0.071               0.408                 0.642   -0.521            0.234               True         True
disp     iaea  0.5       0.684         0.268               0.316                 0.420   -0.416            0.104               True         True
disp     iaea  0.6       0.545         0.051               0.455                 0.593   -0.494            0.138               Tru

## 6. The panels: Fig. 5 sources and the Supplementary Fig. 14 panels

**Display rule for KTW1 (2026-09-19 ruling).** Every condition is drawn in the panels, the high
gas price included, so the reader sees the whole range; the registered verdict is unchanged
(w09 keeps KTW1 for `gashi`, its rows keep `restated = False`, and quoted numbers for it come from
the exact anchors only).

- **y03** (Fig. 5b): per target trajectory, the share of cost samples credit-feasible from each
  online year on at the 50% ceiling (display allowance, band centre), one line per condition. The
  curves only rise.
- **y05** (Fig. 5a): the median required Equivalent ITC per condition, in percent, with the 50%
  statutory maximum dotted. **y06** is the legend shared by y05 and y03.
- **y04** (Supplementary Fig. 14a, b): Table 2 per condition as heat tables: the share never
  reached by the 50% credit, and the share reached by 2038.
- **y02** (Supplementary Fig. 14c): the reference case's predicted shadow-price fan (5th to 95th
  and 25th to 75th percentiles, black median) with each condition's median, in 2024 $/kW on the
  financed basis.
- **y03_mask_perworld_si** (Supplementary Fig. 14d): the y03 drawing authored at the type size of
  the Supplementary composite (canvas width 12.99 in), so Fig. 5 keeps its own type size.

In [7]:
PLOT_W = list(WORLDS7)          # 2026-09-19 (Ethan's ruling): every condition is drawn, KTW1 overridden for the DISPLAY only
# (the registered verdict stands: KTW1 fires for the high-gas-price world, w09; its restated flag stays False in w03-w07)
assert KTW1_WORLDS == ["gashi"], KTW1_WORLDS
# y02 -- the predicted shadow price per world: the base fan (p05-p95, p25-p75) and each world's median
SF14_CW = 12.99                          # 09-24 (E11): Supplementary Fig. 14 is
PS14 = ps.fonts(SF14_CW)                 # height-capped and prints 5.86 in wide
fig, axes = plt.subplots(2, 3, figsize=(7.32, 4.32), sharey=True)
for ax, ab in zip(axes.flat, SCHEDULES):
    b = w07[(w07["world"] == "base") & (w07["schedule"] == ab)].sort_values("year")
    ts = b["year"].to_numpy()
    ax.fill_between(ts, b["p05"], b["p95"], color=ps.SCHED_C[ab], alpha=0.10, lw=0)
    ax.fill_between(ts, b["p25"], b["p75"], color=ps.SCHED_C[ab], alpha=0.20, lw=0)
    ax.plot(ts, b["p50"], color="k", lw=1.6)
    for w in PLOT_W:
        s = w07[(w07["world"] == w) & (w07["schedule"] == ab)].sort_values("year")
        ax.plot(s["year"], s["p50"], color=SENS_C[w], lw=1.0, ls="-" if w != "co2cap" else "--")
    ax.axvline(W_COMMON + 0.5, color="k", lw=0.8, ls="--")
    ax.set_title(ps.SCHED_SHORT[ab], fontsize=PS14["panel_title"]); ax.set_xlim(MODEL_YEARS[0] - 0.95, MODEL_YEARS[-1] + 0.95); ax.xaxis.set_major_locator(MaxNLocator(nbins=3, integer=True)); ax.tick_params(labelsize=PS14["tick"])
fig.supxlabel("build year", fontsize=PS14["axis_label"])
fig.supylabel("Required support, 2024 $/kW\n(financed basis)", fontsize=PS14["axis_label"])
handles = [Line2D([], [], color="k", lw=1.6, label="median sample, reference case"), Patch(facecolor=GREY, alpha=0.25, label="base: 25th–75th"),
           Patch(facecolor=GREY, alpha=0.10, label="base: 5th–95th")] + [Line2D([], [], color=SENS_C[w], lw=1.0, ls="-" if w != "co2cap" else "--", label=f"median, {WLAB[w]}") for w in PLOT_W]
fig.tight_layout(); fig.legend(handles=handles, ncol=2, fontsize=PS14["legend"], frameon=False, loc="upper center", bbox_to_anchor=(0.5, 0.0))
ps.savefig(fig, FIGURES / "y02_shadow_price_fans_perworld.png"); plt.close(fig)

# y03 -- share of cost worlds credit-feasible from each build year on at the 50% cap, per world (display cut, base band; v2.9.1 cumulative)
PF = ps.fonts(ps.PAPER_CANVAS_IN["fig5"])                 # 09-24 (E11): one type spec for every main figure
fig, axes = plt.subplots(2, 3, figsize=(8.4, 5.0), sharey=True)
for ax, ab in zip(axes.flat, SCHEDULES):
    for w in ["base"] + PLOT_W:
        sub = w04c[(w04c["world"] == w) & (w04c["kind"] == "per_year") & (w04c["schedule"] == ab) & (w04c["band"] == "base") & (w04c["cap"] == 0.5)].sort_values("year")
        ax.plot(sub["year"], (1 - sub["share_above_from"]) * 100, color=SENS_C[w], lw=1.6 if w == "base" else 1.0, ls="-" if w != "co2cap" else "--", label=WLAB[w])   # v2.9.1: cumulative
    ax.set_ylim(0, 100); ax.set_title(ps.SCHED_LABEL[ab], fontsize=PF["panel_title"]); ax.set_xlim(MODEL_YEARS[0] - 0.95, MODEL_YEARS[-1] + 0.95); ax.tick_params(labelsize=PF["tick"]); ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
for ax in axes[1]:
    ax.set_xlabel("Online Year", fontsize=PF["axis_label"])               # 09-23 (C21/C3): no demonstration-window marker
for ax in axes[:, 0]:
    ax.set_ylabel("Share of Samples\nCredit-Feasible from\nThat Year On,\n50% Cap (%)", fontsize=PF["axis_label"])
fig.tight_layout()                       # 09-19: no legend here; y06 is the legend shared with y05 (paper Fig 6)
ps.savefig(fig, FIGURES / "y03_mask_perworld.png"); plt.close(fig)

# y04 -- Table 2 per world as a heat table (50% cap, display cut, base band): never credit-feasible share, and under the cap by 2038
fig, axes = plt.subplots(1, 2, figsize=(13.0, 2.95), sharey=True, gridspec_kw=dict(wspace=0.06))
for ax, tbl0, title in ((axes[0], t2, "share never covered\nby the 50% credit"), (axes[1], t2u, "share covered by 2038\n(common window)")):
    PW = ["base"] + sorted(PLOT_W, key=lambda w: float(tbl0[w].mean()))   # 09-19 (Ethan): lowest to highest mean share, reference case first
    tbl = tbl0[PW]
    im = ax.imshow(tbl.to_numpy(float), cmap=ps.CMAP_SEQ, vmin=0, vmax=1, aspect="auto")
    ax.grid(False)                                   # the house grid would draw crosses through the cells
    ax.set_xticks(range(len(PW)), [WLAB[w] for w in PW], rotation=35, ha="right", fontsize=PS14["tick"])
    ax.set_yticks(range(len(SCHEDULES)), [ps.SCHED_SHORT[ab] for ab in SCHEDULES], fontsize=PS14["tick"])
    ax.tick_params(length=0)
    _cm = plt.get_cmap(ps.CMAP_SEQ)
    for i in range(len(SCHEDULES)):
        for j in range(len(PW)):
            v = float(tbl.iloc[i, j])
            r_, g_, b_, _ = _cm(v)
            lum = 0.299 * r_ + 0.587 * g_ + 0.114 * b_        # text colour from the cell's own luminance
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=PS14["annotation"], color="w" if lum < 0.5 else "k")
    ax.set_title(title, fontsize=PS14["panel_title"])
_cb = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
_cb.set_label("share of samples", fontsize=PS14["axis_label"])
_cb.ax.tick_params(labelsize=PS14["tick"])
axes[1].tick_params(axis="y", length=0)
ps.letter_panels(axes, size=PS14["panel_letter"]); ps.savefig(fig, FIGURES / "y04_table2_perworld.png"); plt.close(fig)
# y03 (Supplementary twin) -- the same drawing as y03 above, authored for Supplementary Fig. 14's
# composed canvas instead of main-text Fig. 5's. 09-24 (E11, `_edit_20260924_si_type_market_transfer.py`):
# one PNG cannot carry two authored type sizes, and this panel appears in both figures.
fig, axes = plt.subplots(2, 3, figsize=(8.00, 5.00), sharey=True)
for ax, ab in zip(axes.flat, SCHEDULES):
    for w in ["base"] + PLOT_W:
        sub = w04c[(w04c["world"] == w) & (w04c["kind"] == "per_year") & (w04c["schedule"] == ab) & (w04c["band"] == "base") & (w04c["cap"] == 0.5)].sort_values("year")
        ax.plot(sub["year"], (1 - sub["share_above_from"]) * 100, color=SENS_C[w], lw=1.6 if w == "base" else 1.0, ls="-" if w != "co2cap" else "--", label=WLAB[w])
    ax.set_ylim(0, 100); ax.set_title(ps.SCHED_LABEL[ab].replace(" (", "\n("), fontsize=PS14["panel_title"]); ax.set_xlim(MODEL_YEARS[0] - 0.95, MODEL_YEARS[-1] + 0.95); ax.tick_params(labelsize=PS14["tick"]); ax.xaxis.set_major_locator(MaxNLocator(nbins=3, integer=True))
fig.supxlabel("Online Year", fontsize=PS14["axis_label"])
fig.supylabel("Share of Samples Credit-Feasible\nfrom That Year On, 50% Cap (%)", fontsize=PS14["axis_label"])
fig.tight_layout()
ps.savefig(fig, FIGURES / "y03_mask_perworld_si.png"); plt.close(fig)
print("y02-y04 written, plus the y03 Supplementary twin y03_mask_perworld_si.png")

saved -> analysis\results\exports\11\y02_shadow_price_fans_perworld.png


saved -> analysis\results\exports\11\y03_mask_perworld.png


saved -> analysis\results\exports\11\y04_table2_perworld.png


saved -> analysis\results\exports\11\y03_mask_perworld_si.png
y02-y04 written, plus the y03 Supplementary twin y03_mask_perworld_si.png


In [8]:
# y05 -- the paper's Fig 4a per world (Ethan, 2026-09-18): the base world's rate fan in percent (display twin, clipped at
# 100%) with the median required rate of each restated market world drawn over it; the y02 overlay in the credit-rate unit
def _clip_spans(ts, top, lim=1.0):
    ts, top = np.asarray(ts, float), np.asarray(top, float)
    spans, x0 = [], None
    for j in range(len(ts)):
        if top[j] > lim and x0 is None:
            x0 = ts[j] if j == 0 else ts[j-1] + (lim - top[j-1]) / (top[j] - top[j-1]) * (ts[j] - ts[j-1])
        elif top[j] <= lim and x0 is not None:
            spans.append((x0, ts[j-1] + (lim - top[j-1]) / (top[j] - top[j-1]) * (ts[j] - ts[j-1])))
            x0 = None
    if x0 is not None:
        spans.append((x0, ts[-1]))
    return spans

PF = ps.fonts(ps.PAPER_CANVAS_IN["fig5"])                 # 09-24 (E11): one type spec for every main figure
fig, axes = plt.subplots(2, 3, figsize=(8.4, 5.0), sharey=True)
for ax, ab in zip(axes.flat, SCHEDULES):
    b = w03c[(w03c["world"] == "base") & (w03c["schedule"] == ab) & (w03c["band"] == "base") & (w03c["quantity"] == "rate")].sort_values("year")
    ts = b["year"].to_numpy()
    ax.plot(ts, b["p50"], color=SENS_C["base"], lw=1.6)        # 09-19: medians only (Ethan); the base bands are in Fig 4a
    for w in PLOT_W:
        s_ = w03c[(w03c["world"] == w) & (w03c["schedule"] == ab) & (w03c["band"] == "base") & (w03c["quantity"] == "rate")].sort_values("year")
        ax.plot(s_["year"], s_["p50"], color=SENS_C[w], lw=1.0, ls="-" if w != "co2cap" else "--")
    ax.axhline(CAPS[0], color=ps.ACCENT["red"], lw=0.9, ls=":")   # 09-23 (C21): no 60% ceiling, no window marker
    ax.set_ylim(0.0, 1.05)   # 09-24 (Ethan F3): the 2025 EO low-gas median reaches 100.6% in 2050;
                             # at a 100% limit it was cut off at the top edge and read as exactly 100%
    ax.set_title(ps.SCHED_LABEL[ab], fontsize=PF["panel_title"])
    ax.set_xlim(MODEL_YEARS[0] - 0.95, MODEL_YEARS[-1] + 0.95)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.tick_params(labelsize=PF["tick"])
for ax in axes[1]:
    ax.set_xlabel("Online Year", fontsize=PF["axis_label"])
for ax in axes[:, 0]:
    ax.set_ylabel("Equivalent ITC (%)", fontsize=PF["axis_label"])        # 09-23 (C21): the same words as Fig 4
fig.tight_layout()                       # 09-19: no legend here; y06 is the legend shared with y03 (paper Fig 6)
ps.savefig(fig, FIGURES / "y05_rate_fans_perworld_pct.png")
plt.close(fig)

# y06 -- the legend shared by y05 (Fig 5a) and y03 (Fig 5b): one entry per market condition (the lines are medians
# in y05 and cumulative shares in y03) and the 50% statutory maximum (09-23, C21: the 60% cap and the window end are gone)
PF = ps.fonts(ps.PAPER_CANVAS_IN["fig5"])                 # 09-24 (E11): one type spec for every main figure
fig = plt.figure(figsize=(8.4, 1.1))   # 09-24: three rows of three at the paper type size
handles = [Line2D([], [], color=SENS_C[w], lw=1.6 if w == "base" else 1.0, ls="-" if w != "co2cap" else "--", label=WLAB[w]) for w in ["base"] + PLOT_W]
handles += [Line2D([], [], color=ps.ACCENT["red"], lw=0.9, ls=":", label="current statutory maximum 50%")]
fig.legend(handles=handles, ncol=3, fontsize=PF["legend"], frameon=False, loc="center")
ps.savefig(fig, FIGURES / "y06_fig6_legend.png")
plt.close(fig)
print("y05, y06 written")

saved -> analysis\results\exports\11\y05_rate_fans_perworld_pct.png


saved -> analysis\results\exports\11\y06_fig6_legend.png
y05, y06 written


### Supplementary Fig. 14

The composite stacks y04 (internally lettered a, b), y02 (c) and y03_mask_perworld_si (d), one
panel per row, centred, at the panels' native pixels. The stamped letters use the paper's panel
letter size of the published figure (84 px at 300 dpi, the letter size for a canvas 13.10 in wide;
the panels themselves are authored at 12.99 in).

In [9]:
from nlpaper import compose as C
# The panels are authored at 12.99 in (SF14_CW above). The stamped letters of the published
# Supplementary Fig. 14 are 84 px (20.16 pt at 300 dpi); fonts(12.99) gives 83 px. The letters of
# the published image were stamped at a slightly wider canvas; 13.10 in is the width whose letter
# size reproduces the published image pixel for pixel (searched over +-6 px around 83 px).
SF14_LETTER_CW = 13.10
_ = C.compose([[FIGURES / "y04_table2_perworld.png"],
               [FIGURES / "y02_shadow_price_fans_perworld.png"],
               [FIGURES / "y03_mask_perworld_si.png"]],
              "SF14_market_requirement_transfer.png", letters=["", "c", "d"], center=True,
              letter_pt=ps.fonts(SF14_LETTER_CW)["panel_letter"], out_dir=paths.FIGS_SI, show=False)

composed -> analysis\results\figures\si\SF14_market_requirement_transfer.png  (3902x5192 px)


## 7. w09: the verdict

`w09` collects every gate, kill, test and quoting disposition registered for this analysis:
R0, G0c, GT0, GA-w, GT0b, GT1/GT2 per condition (all PARTIAL), KTW1 (fires for the high gas price)
and KTW2 (fires nowhere), EC (passes), the status flips, the no-shift pairs, the slack-rule record,
the 2038 envelope and the map basis. Notebook 16 prints it as Supplementary Table 4g.

In [10]:
VERDICT.append(("claim", "K1", "STANDS", "block A is quoted as solved-anchor counts and as the map's population shares per world, never as an in-sample capture number"))
VERDICT.append(("basis", "map-basis", CHOSEN, f"v2.9 map ({CHOSEN}); reference m109; bands E7 {E7_PTS} combined in quadrature with each world's leave-one-out q90; "
                f"not returned: {NOT_RETURNED} (demhi on 10 anchors); stems batch3_"))
w09 = pd.DataFrame(VERDICT, columns=["kind", "name", "result", "detail"])
w09.to_csv(EXPORTS / "w09_verdict.csv", index=False)
print(w09.to_string(index=False))
from datetime import datetime, timezone
meta = {"notebook": "11_market_transfer.ipynb", "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "registration": "market_transfer methods v2.0 (2026-09-18) on rate_design methods v2.6 + v2.9",
        "map": CHOSEN, "reference": "m109", "E7_pts": E7_PTS, "E7w_pts": E7W, "q90_world": Q90, "gates": {w: GT[w]["verdict"] for w in WORLDS7},
        "ktw1": KTW1_WORLDS, "ktw2": [w for w in WORLDS7 if GT[w]["ktw2"]], "EC": EC, "no_shift": NO_SHIFT, "not_returned": NOT_RETURNED}
(EXPORTS / "market_transfer_metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("metadata written")

   kind                name             result                                                                                                                                                                                                                                                                                                                                                                                  detail
   gate                  R0               PASS                                                                                                                                                                                                                                                                                                                      rate_design2_b3_checks registry: {'PASS': 45, 'INFO': 10}; no FAIL
   gate                 G0c               PASS                                                                                                            

## 8. Who reads these outputs

- `w01_required_rate_perworld.csv`: notebooks 12_flat_credit (Supplementary Fig. 15) and
  15_new_plant_value (Supplementary Fig. 26, Table 17), and notebook 16 (Supplementary Table 4f).
- `w02`, `w04_cut`, `w06`, `w07`, `w08`, `w09`: notebook 16 writes Supplementary Table 4 a to g.
- `y05`, `y03`, `y06`: notebook 16 composes Fig. 5.
- `results/figures/si/SF14_market_requirement_transfer.png` is written here directly.

In [11]:
for p in sorted(EXPORTS.glob("*")):
    print(f"{paths.rel(p)}  {p.stat().st_size:,} bytes")

analysis\results\exports\11\market_transfer_metadata.json  1,893 bytes
analysis\results\exports\11\w00_status_flips.csv  1,989 bytes
analysis\results\exports\11\w01_required_rate_perworld.csv  486,860 bytes
analysis\results\exports\11\w02_world_year_shift.csv  15,724 bytes
analysis\results\exports\11\w02b_shift_loo_cells.csv  337,547 bytes
analysis\results\exports\11\w02c_band_e7w.csv  2,993 bytes
analysis\results\exports\11\w03_requirement_fans_perworld.csv  288,768 bytes
analysis\results\exports\11\w03_requirement_fans_perworld_cut.csv  288,531 bytes
analysis\results\exports\11\w04_feasibility_mask_perworld.csv  227,474 bytes
analysis\results\exports\11\w04_feasibility_mask_perworld_cut.csv  227,472 bytes
analysis\results\exports\11\w05_demonstration_gap_perworld.csv  71,697 bytes
analysis\results\exports\11\w05_demonstration_gap_perworld_cut.csv  71,483 bytes
analysis\results\exports\11\w06_common_window_perworld.csv  55,834 bytes
analysis\results\exports\11\w07_shadow_price_fans_pe